In [9]:
import pandas as pd
import re

# 1. 파일 불러오기
df_yanolja = pd.read_excel("nol_reviews.xlsx")
df_naver = pd.read_excel("naver_reviews.xlsx")
df_yeogi = pd.read_excel("yeogi_reviews.xlsx")

# 2. 플랫폼 컬럼 추가
df_yanolja['플랫폼'] = '야놀자'
df_naver['플랫폼'] = '네이버'
df_yeogi['플랫폼'] = '여기어때'

# 3. 날짜 형식 전처리
def fix_naver_date(date_str):
    nums = re.findall(r'\d+', str(date_str))
    if len(nums) == 2:
        return f"2026-{int(nums[0]):02d}-{int(nums[1]):02d}"
    elif len(nums) >= 3:
        year = int(nums[0])
        year = year + 2000 if year < 100 else year 
        return f"{year}-{int(nums[1]):02d}-{int(nums[2]):02d}"
    return pd.NaT 

def fix_normal_date(date_str):
    nums = re.findall(r'\d+', str(date_str))
    if len(nums) >= 3:
        return f"{nums[0]}-{int(nums[1]):02d}-{int(nums[2]):02d}"
    return pd.NaT

# 함수 적용 및 완벽한 날짜 데이터(datetime)로 변환
df_naver['작성일자'] = pd.to_datetime(df_naver['작성일자'].apply(fix_naver_date))
df_yanolja['작성일자'] = pd.to_datetime(df_yanolja['작성일자'].apply(fix_normal_date))
df_yeogi['작성일자'] = pd.to_datetime(df_yeogi['작성일자'].apply(fix_normal_date))

# 4. 세 데이터프레임 통합
df_merged = pd.concat([df_yanolja, df_naver, df_yeogi], ignore_index=True)

# 5. 결측치(빈칸) 및 10자 미만 무성의 리뷰 제거
df_merged = df_merged.dropna(subset=['리뷰내용', '작성일자'])
df_merged = df_merged[df_merged['리뷰내용'] != '내용 없음']
df_merged['리뷰내용'] = df_merged['리뷰내용'].astype(str)

min_length = 20
df_final = df_merged[df_merged['리뷰내용'].str.len() >= min_length]

# 6. 최신순 정렬 (날짜 형태 변환 전에 정렬해야 정확합니다)
df_final = df_final.sort_values(by='작성일자', ascending=False)

# 💡 변경된 부분: 'YYYY년 M월 D일' 형식으로 변환 (앞에 0이 붙지 않도록 처리)
df_final['작성일자'] = df_final['작성일자'].apply(lambda x: f"{x.year}년 {x.month}월 {x.day}일")

# 보기 좋게 컬럼 순서 정렬
df_final = df_final[['플랫폼', '작성일자', '아이디', '객실명', '리뷰내용']]

# 7. 최종 엑셀 파일 저장
df_final.to_excel("all_platforms_reviews.xlsx", index=False)

print("✅ 데이터 병합 및 '0000년 0월 0일' 형식 변환 완료!")
print(f"최종 유효 리뷰: 총 {len(df_final)}건")

✅ 데이터 병합 및 '0000년 0월 0일' 형식 변환 완료!
최종 유효 리뷰: 총 2497건
